# L8 · Notebook 02 — Baird's 反例：Deadly Triad 的实证发散

**对应 PPT**：`L8-VFA.tex` §3 Deadly Triad / Baird's counterexample（1995）

## 教学目标

**Deadly Triad** = ① 函数逼近 + ② bootstrapping + ③ off-policy 训练，三者同时出现就**可能**发散。
Baird 1995 给出的 7-state MDP 是经典构造：理论可证半梯度 TD 的权重以指数速率发散。

1. 复现 Baird's 7-state MDP（Sutton-Barto 第 11.2 节）
2. 让权重在 off-policy 半梯度 TD 下**发散到 $\infty$**
3. 切换到 on-policy 重跑，看权重稳定（验证 deadly triad 必须三件齐）
4. 直观感受为什么 deep RL 早期『训练崩』是规律不是意外

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## 1. Baird's 7-state MDP

- 状态 `0..6`（7 个）；动作 2 个：`solid`（→ state 6）/ `dashed`（→ uniform random 0..5）
- 奖励恒为 0；折扣 $\gamma = 0.99$
- **目标策略** $\pi$：always `solid` （即恒去 state 6）
- **行为策略** $b$：1/7 概率 `solid`，6/7 概率 `dashed`（保证 6 个非目标态被访问）
- **特征** $\phi(s)\in\mathbb R^8$（Sutton-Barto Fig 11.1 给出）；权重 $w\in\mathbb R^8$
- 真值 $v^\pi \equiv 0$（因为 reward 恒 0），所以投影/不动点都应是 $w=0$

In [ ]:
# Sutton-Barto Fig 11.1 的特征：state s ∈ {0..5} 给 phi[s]=2, phi[7]=1；state 6 给 phi[6]=1, phi[7]=2
PHI = np.zeros((7, 8))
for s in range(6):
    PHI[s, s] = 2.0
    PHI[s, 7] = 1.0
PHI[6, 6] = 1.0
PHI[6, 7] = 2.0
print('Φ 矩阵 (7×8):')
print(PHI)
print('注意：v^π ≡ 0 在 Φ 张成空间内（取 w=0 即可）— 投影解存在。')

In [ ]:
def step(s, action):
    """action: 0=dashed (uniform 0..5), 1=solid (→6).  奖励恒 0；无终止态"""
    if action == 1:
        s2 = 6
    else:
        s2 = rng.integers(0, 6)
    return s2, 0.0

def importance_ratio(action_b, action_pi):
    """目标策略 always 1；行为策略 6/7 dashed (0), 1/7 solid (1)"""
    pi = 1.0 if action_pi == 1 else 0.0
    b  = 1/7 if action_b  == 1 else 6/7
    return pi / b

## 2. Off-policy 半梯度 TD：观察发散

$$w \leftarrow w + \alpha\,\rho\,\bigl(r + \gamma\,\phi(s')^\top w - \phi(s)^\top w\bigr)\,\phi(s)$$

$\rho = \pi(a|s)/b(a|s)$ 是重要性比。

In [ ]:
def run_baird(off_policy=True, n_steps=1000, alpha=0.01, gamma=0.99, seed=0):
    global rng
    rng = np.random.default_rng(seed)
    # 初始 w：从 Sutton-Barto Fig 11.2 经典初值
    w = np.array([1, 1, 1, 1, 1, 1, 10, 1.0])
    history = [w.copy()]
    s = rng.integers(0, 7)
    for t in range(n_steps):
        if off_policy:
            # 行为策略：1/7 solid, 6/7 dashed
            a_b = 1 if rng.random() < 1/7 else 0
            a_pi = 1  # 目标策略
            rho = importance_ratio(a_b, a_pi)
        else:
            a_b = 1  # 直接按目标策略
            rho = 1.0
        s2, r = step(s, a_b)
        td = r + gamma * PHI[s2] @ w - PHI[s] @ w
        w = w + alpha * rho * td * PHI[s]
        history.append(w.copy())
        s = s2
    return np.array(history)

hist_off = run_baird(off_policy=True,  n_steps=1000, alpha=0.01)
hist_on  = run_baird(off_policy=False, n_steps=1000, alpha=0.01)

print(f'off-policy 1000 步后 ||w|| = {np.linalg.norm(hist_off[-1]):.3e}')
print(f'on-policy  1000 步后 ||w|| = {np.linalg.norm(hist_on[-1]):.3e}')
print('真值 ||w*|| = 0   （v^π ≡ 0）')

## 3. 可视化：8 个权重分量随时间

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, hist, title in [
    (axes[0], hist_off, 'Off-policy: 三件齐 → 发散'),
    (axes[1], hist_on,  'On-policy: 撤掉一件 → 稳定'),
]:
    for i in range(8):
        ax.plot(hist[:, i], label=f'$w_{i+1}$', alpha=0.8)
    ax.set_xlabel('step'); ax.set_ylabel('权重值')
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=2, loc='upper left')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/baird_divergence.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# log-scale 范数曲线：发散是指数级
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(np.linalg.norm(hist_off, axis=1), label='off-policy ||w||', linewidth=1.5)
ax.semilogy(np.linalg.norm(hist_on,  axis=1), label='on-policy  ||w||', linewidth=1.5)
ax.set_xlabel('step'); ax.set_ylabel(r'$\|w\|_2$ (log scale)')
ax.set_title('Deadly Triad 实证：off-policy 半梯度 TD 指数发散')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('figures/baird_norm.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. 课堂诊断小结

| 配置 | 1000 步后 $\|w\|$ | 行为 |
|---|---|---|
| 函数逼近 + bootstrap + **off-policy** | 数量级 $10^k, k\gg 0$ | 指数发散 |
| 函数逼近 + bootstrap + **on-policy** | $\le\|w_0\|$ | 稳定（Tsitsiklis-Van Roy） |

**关键观察**：真值 $v^\pi \equiv 0$ 在张成空间内（取 $w=0$ 即可），所以**没有逼近误差**；发散纯属
算法不稳定，不是表示能力不够。

## 思考题

1. DQN 同时用了函数逼近、bootstrap、off-policy（来自 replay），按 Baird 论断『必然发散』。为什么实际能跑？
   （提示：target network 把 bootstrap 部分变成『伪监督学习』 → §4 帧）
2. 把 $\gamma$ 改小到 0.5，发散速度变化如何？$\gamma=0$ 呢？
3. 真梯度（full gradient TD / GTD2）能避免这个发散吗？代价是什么？